In [78]:
from bs4 import BeautifulSoup
import requests
import folium
import pandas as pd
import re
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from urllib.parse import urlparse, parse_qs
from sklearn.model_selection import train_test_split
from folium.plugins import HeatMap

In [79]:
!apt-get update -q
!apt-get install -y -q google-chrome-stable
!wget https://chromedriver.storage.googleapis.com/113.0.5672.63/chromedriver_linux64.zip
!unzip chromedriver_linux64.zip
!mv chromedriver /usr/bin/chromedriver
!chmod +x /usr/bin/chromedriver

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.
'mv' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.


In [80]:
!pip install geopy

In [ ]:
!apt-get update -y
!apt-get install -y google-chrome-stable
!apt-get install -y chromedriver
!pip install selenium
!pip install webdriver-manager

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# List untuk menyimpan semua data properti yang di-scrape
data_rentals = []

# Loop melalui halaman 1 hingga 6
for page in range(1, 7):
    # Menentukan URL berdasarkan nomor halaman
    if page == 1:
        url = "https://www.bukitvista.com/search-results?location%5B%5D=&areas%5B%5D=&bedrooms="
    else:
        url = f"https://www.bukitvista.com/search-results/page/{page}?location%5B%5D=&areas%5B%5D=&bedrooms="

    # Mengambil halaman web dengan requests
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36'}
    response = requests.get(url, headers=headers)

    # Membaca kode HTML dari halaman yang diambil
    soup = BeautifulSoup(response.text, 'html.parser')

    # Mencari semua elemen properti
    rentals = soup.find_all('div', class_='item-body flex-grow-1')

    # Mengambil data dari setiap elemen properti
    for content in rentals:
        try:
            name = content.find('h2', class_='item-title').text.strip()
            location = content.find('address', class_='item-address').text.strip()
            bedrooms = content.find('li', class_='h-beds').text.strip() if content.find('li', class_='h-beds') else 'N/A'
            bathrooms = content.find('li', class_='h-baths').text.strip() if content.find('li', class_='h-baths') else 'N/A'
            property_type = content.find('li', class_='h-type').text.strip() if content.find('li', class_='h-type') else 'N/A'
            price = content.find('ul', class_='item-price-wrap hide-on-list').text.strip() if content.find('ul', class_='item-price-wrap hide-on-list') else 'N/A'
            agent = content.find('div', class_='item-author').text.strip() if content.find('div', class_='item-author') else 'N/A'
            date = content.find('div', class_='item-date').text.strip() if content.find('div', class_='item-date') else 'N/A'

            # Mengambil URL dari setiap properti
            url_element = content.find('a', attrs={'target': '_self'})
            url = url_element['href'] if url_element and 'href' in url_element.attrs else 'N/A'

            # Menyimpan semua informasi yang berhasil didapatkan ke dalam list data_rentals
            data_rentals.append({
                'name': name,
                'location': location,
                'bedrooms': bedrooms,
                'bathrooms': bathrooms,
                'property_type': property_type,
                'price_range': price,
                'url': url,
                'agency': agent,
                'date': date
            })
        except AttributeError:
            continue  # Lewati jika ada elemen yang tidak ditemukan

# Konversi ke DataFrame
df = pd.DataFrame(data_rentals)
pd.set_option('display.max_columns', None)
df.head()


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Inisialisasi geolocator dengan user_agent yang sesuai
geolocator = Nominatim(user_agent="bv_scraper", timeout=10)

# Gunakan RateLimiter untuk membatasi frekuensi permintaan
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=2.0)

# List untuk menyimpan semua data properti yang di-scrape
data_rentals = []


In [ ]:
for page in range(1, 7):  # Hanya sampai halaman 6
    if page == 1:
        url = "https://www.bukitvista.com/search-results?location%5B%5D=&areas%5B%5D=&bedrooms="
    else:
        url = f"https://www.bukitvista.com/search-results/page/{page}?location%5B%5D=&areas%5B%5D=&bedrooms="

    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    rentals = soup.find_all('div', class_='item-body flex-grow-1')

    for content in rentals:
        try:
            name = content.find('h2', class_='item-title').text.strip()
            location = content.find('address', class_='item-address').text.strip()
            bedrooms = content.find('li', class_='h-beds').text.strip() if content.find('li', class_='h-beds') else 'N/A'
            bathrooms = content.find('li', class_='h-baths').text.strip() if content.find('li', class_='h-baths') else 'N/A'
            property_type = content.find('li', class_='h-type').text.strip() if content.find('li', class_='h-type') else 'N/A'
            price = content.find('ul', class_='item-price-wrap hide-on-list').text.strip() if content.find('ul', class_='item-price-wrap hide-on-list') else 'N/A'
            agent = content.find('div', class_='item-author').text.strip() if content.find('div', class_='item-author') else 'N/A'
            date = content.find('div', class_='item-date').text.strip() if content.find('div', class_='item-date') else 'N/A'

            url_element = content.find('a', attrs={'target': '_self'})
            detail_url = url_element['href'] if url_element and 'href' in url_element.attrs else 'N/A'

            data_rentals.append({
                'name': name,
                'location': location,
                'bedrooms': bedrooms,
                'bathrooms': bathrooms,
                'property_type': property_type,
                'price_range': price,
                'agency': agent,
                'date': date,
                'url': detail_url
            })
        except AttributeError:
            continue


In [ ]:
detailed_data_rentals = []

for rental in data_rentals:
    detail_url = rental['url']
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(detail_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    try:
        detailed_address = soup.find('li', class_='detail-address').find('span').text.strip() if soup.find('li', class_='detail-address') else None
        city = soup.find('li', class_='detail-city').find('span').text.strip() if soup.find('li', class_='detail-city') else None
        state = soup.find('li', class_='detail-state').find('span').text.strip() if soup.find('li', class_='detail-state') else None
        area = soup.find('li', class_='detail-area').find('span').text.strip() if soup.find('li', class_='detail-area') else None
        country = soup.find('li', class_='detail-country').find('span').text.strip() if soup.find('li', class_='detail-country') else None
        zip_code = soup.find('li', class_='detail-zip').find('span').text.strip() if soup.find('li', class_='detail-zip') else None
        number_of_guests = soup.find('li', class_='guest-number').find('span').text.strip() if soup.find('li', class_='guest-number') else None
        property_status = soup.find('li', class_='prop_status').find('span').text.strip() if soup.find('li', class_='prop_status') else None

        # Airbnb link
        airbnb_link = soup.find('a', class_=[
            'wp-block-button__link has-vivid-red-background-color has-background wp-element-button',
            'wp-block-button__link has-white-color has-vivid-red-background-color has-text-color has-background wp-element-button',
            'wp-block-button__link has-vivid-red-background-color has-background has-text-align-left wp-element-button',
            'wp-block-button__link has-vivid-red-background-color has-background',
            'wp-block-button__link has-white-color has-vivid-red-background-color has-text-color has-background'
        ])
        airbnb_url = airbnb_link['href'] if airbnb_link and airbnb_link.has_attr('href') else None

        # Google Maps link
        maps_link = None
        for a in soup.find_all('a', href=True, class_='btn btn-primary btn-slim'):
            if 'maps.google.com' in a['href']:
                maps_link = a
                break
        maps_url = maps_link['href'] if maps_link else None

        # Koordinat dari lokasi
        latitude, longitude = None, None
        full_address = rental['location']

        if full_address and full_address != 'N/A':
            try:
                location = geocode(full_address)
                if location:
                    latitude = location.latitude
                    longitude = location.longitude
            except (GeocoderTimedOut, GeocoderServiceError) as e:
                print(f"Error geocoding lokasi '{full_address}': {e}")

        # Tambahkan data ke list
        detailed_data_rentals.append({
            'name': rental['name'],
            'location': rental['location'],
            'bedrooms': rental['bedrooms'],
            'bathrooms': rental['bathrooms'],
            'property_type': rental['property_type'],
            'price_range': rental['price_range'],
            'url': rental['url'],
            'agency': rental.get('agency', 'N/A'),
            'detailed_address': detailed_address,
            'city': city,
            'state': state,
            'area': area,
            'country': country,
            'zip_code': zip_code,
            'number_of_guests': number_of_guests,
            'property_status': property_status,
            'maps_url': maps_url,
            'latitude': latitude,
            'longitude': longitude,
            'airbnb_url': airbnb_url
        })

    except AttributeError:
        continue


In [ ]:
df_detailed_rentals = pd.DataFrame(detailed_data_rentals)
pd.set_option('display.max_columns', None)
df_detailed_rentals.head()


In [ ]:
# menyimpan file csv
#df_detailed_rentals.to_csv('bukitvista_detailed_rentals.csv', index=False)


untuk airbnb

In [ ]:
import requests
# Check whether targeted URL allows scraping
urlbnb = 'https://www.airbnb.com/'
response = requests.get(urlbnb)
# Check if the request status was successful (status code = 200)
if response.status_code == 200:
  print("Website is accessible")
  # Check for specific headers that might indicate scraping is disallowed
  if "X-Robots-Tag" in response.headers:
    if "noindex" in response.headers["X-Robots-Tag"] or "nofollow" in response.headers["X-Robots-Tag"]:
      print("Scraping might be disallowed based on X-Robots-Tag header.")
  else:
    print("No X-Robots-Tag header found.")
else:
  print(f"Website is not accessible. Status code: {response.status_code}")

In [ ]:
urlbnb = 'https://www.airbnb.com' # get url
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"}  # avoid blocking
response = requests.get(urlbnb, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    print(soup.title.text)  # showing title page
else:
    print("failed to access")

In [ ]:
from bs4 import BeautifulSoup

# Membuat objek BeautifulSoup dengan parser 'html'
soup = BeautifulSoup(response.content, 'html')

# Menampilkan struktur HTML yang telah diformat
print(soup.prettify())


In [ ]:
%%time
# Install selenium dan webdriver manager
!pip install selenium webdriver-manager

# Import library
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# Setup Chrome dan Selenium
def setup_driver():
    service = Service(ChromeDriverManager().install())
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # agar berjalan di background
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(service=service, options=options)
    return driver

# Contoh fungsi untuk mengambil rating Airbnb
def get_airbnb_rating(url):
    driver = setup_driver()
    try:
        driver.get(url)
        time.sleep(5)  # tunggu render halaman
        rating_elements = driver.find_elements(By.XPATH, "//div[@aria-hidden='true']")
        print(f"number of element found: {len(rating_elements)}")

        for element in rating_elements:
            print("Rating:", element.text.strip() or "no text")

    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.quit()

# Contoh penggunaan, ganti URL_AIRBNB dengan link Airbnb yang valid
get_airbnb_rating('https://www.airbnb.com/rooms/1201807361024443576?source_impression_id=p3_1741404387_P3bY7wr7jbrT6BU9')
print(get_airbnb_rating)


In [ ]:
%%time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup Selenium
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# define scrap rating
def scrape_airbnb_ratings(df):
    combined_results = []

    for url in df['airbnb_url']:
        try:
            driver.get(url)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(5)

            wait = WebDriverWait(driver, 10)
            wait.until(EC.presence_of_element_located((By.XPATH, "//div[@aria-hidden='true'] | //span[@aria-hidden='true']")))
            # find rating element from different class element 
            element_1 = driver.find_elements(By.XPATH, "//div[@aria-hidden='true'] | //span[@aria-hidden='true']")
            element_2 = driver.find_elements(By.XPATH, "//div[contains(@class, 'r1lutz1s atm_c8_o7aogt atm_c8_l52nlx__oggzyc dir dir-ltr') and @aria-hidden='true']")
            element_3 = driver.find_elements(By.XPATH, "//span[contains(@class, 'a8jt5op atm_3f_idpfg4 atm_7h_hxbz6r atm_7i_ysn8ba atm_e2_t94yts atm_ks_zryt35 atm_l8_idpfg4 atm_mk_stnw88 atm_vv_1q9ccgz atm_vy_t94yts dir dir-ltr') and @aria-hidden='true']")
           
            # combine all element
            all_element = element_1 + element_2 + element_3
            result = None

            # get text from element
            for el in all_element:
                text = el.text.strip()
                if re.match(r'^\d\.\d{1,2}$', text): # get rating format
                    result = text
                    break

            combined_results.append(result)

        except Exception as e:
            combined_results.append(f"Error: {e}")

    driver.quit()
    df['rating'] = combined_results
    return df
df_detailed_rentals = scrape_airbnb_ratings(df_detailed_rentals)
df_detailed_rentals

In [ ]:
# menyimpan file csv
#df_detailed_rentals.to_csv('bukitvista_airbnb.csv', index=False)

In [ ]:
df = pd.read_csv('bukitvista_airbnb.csv')
pd.set_option('display.max_columns', None)
df.head()

## DATA CLEANING

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [56]:
# Daftar kategori yang ingin diekstrak sebagai kolom one-hot
property_keywords = [
    'Amazing View', 'Amazing pool', 'Beachfront', 'Golfing',
    'Guest House', 'Island life', 'Jungle View', 'Ocean view',
    'Pool view', 'Residential', 'Rice paddy view', 'Style',
    'Surfing', 'Tropical', 'View', 'Villa'
]

# Untuk setiap keyword, membuat kolom biner (1 jika keyword ada di property_type, 0 jika tidak)
for keyword in property_keywords:
    df[keyword] = df['property_type'].fillna('').apply(lambda x: int(keyword.lower() in x.lower()))

# Kategori umum properti
def categorize_property_type(ptype):
    if pd.isna(ptype):
        return 'Other'
    elif 'Guest House' in ptype:
        return 'Guest House'
    elif 'Villa' in ptype:
        return 'Villa'
    else:
        return 'Other'
df['property_category'] = df['property_type'].apply(categorize_property_type)
df

,name,location,bedrooms,bathrooms,property_type,price_range,url,agency,detailed_address,city,state,area,country,zip_code,number_of_guests,property_status,maps_url,latitude,longitude,airbnb_url,rating,Amazing View,Amazing pool,Beachfront,Golfing,Guest House,Island life,Jungle View,Ocean view,Pool view,Residential,Rice paddy view,Style,Surfing,Tropical,View,Villa,property_category
0,Uluwatu Modern Boho Villa Near Nyang Nyang Beach,"Jl. Batu Nunggul No.1, Pecatu, Kec. Kuta Sel.,...",Beds: 2,Baths: 2,"Amazing pool, Island life, Pool view, Surfing,...",$258 per 2 nights,https://www.bukitvista.com/property/uluwatu-mo...,Agus Weda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.airbnb.com/rooms/1287975372353817564,4.4,0,1,0,0,0,1,0,0,1,0,0,0,1,1,1,1,Villa
1,Bingin Beach Hideaway: Group Villa with Pool &...,"Jl. Pantai Cemongkak Gg. Samuh Sari No.2, Peca...",Beds: 3,Baths: 3,"Amazing pool, Island life, Pool view, Surfing,...",Starting from USD 161 per night,https://www.bukitvista.com/property/bingin-bea...,Bukit Vista,"Jl. Pantai Cemongkak Gg. Samuh Sari No.2, Peca...",Bali,Bali,Bingin,Indonesia,80361.0,6.0,"Bali Vacation Rental, Group Friendly Stay, Lon...",http://maps.google.com/?q=Jl.%20Pantai%20Cemon...,NaN,NaN,https://www.airbnb.com/rooms/1201807361024443576,NaN,0,1,0,0,0,1,0,0,1,0,0,0,1,1,1,1,Villa
2,4-Bedroom Mediterranean Luxury Villa with Ubud...,"Jl. Suweta, Ubud, Kecamatan Ubud, Kabupaten Gi...",Beds: 4,Baths: 4,"Amazing pool, Amazing View, Jungle View, Pool ...",USD 202 / Night,https://www.bukitvista.com/property/mediterran...,Bukit Vista,"Jl. Suweta, Ubud, Kecamatan Ubud, Kabupaten Gi...",Bali,Bali,Ubud,Indonesia,80571.0,8.0,"Bali Vacation Rental, Group Friendly Stay, Ins...","http://maps.google.com/?q=Jl.%20Suweta,%20Ubud...",-8.500216,115.264779,https://www.airbnb.com/rooms/1338046513545783704,5.0,1,1,0,0,0,0,1,0,1,0,0,0,0,1,1,1,Villa
3,Luxurious 3-Bedroom Nusa Dua Seafront Villa w/...,NaN,Beds: 3,Baths: 4,"Beachfront, Villa",Starting from USD 715 per 2 nights,https://www.bukitvista.com/property/nusa-dua-s...,Konang,"Jl. Raya Nusa Dua Selatan, Sawangan, Nusa Dua,...",Bali,Bali,Nusa Dua,Indonesia,80363.0,6.0,"Bali Vacation Rental, Top Trending",NaN,NaN,NaN,https://www.airbnb.com/rooms/945876?check_in=2...,4.84,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,Villa
4,Grand Villa Retreat w/ Pool & Garden in Ungasan,"Jl. Pantai Balangan I No.9x, Ungasan, Kec. Kut...",Beds: 2,Baths: 2,"Amazing pool, Golfing, Pool view, Villa",Starting from USD 84 per night,https://www.bukitvista.com/property/grand-vill...,Bukit Vista,"Jl. Pantai Balangan I No.9x, Ungasan, Kec. Kut...",Bali,Bali,Balangan,Indonesia,80361.0,4.0,"Group Friendly Stay, Long Stay Trend Property,...",http://maps.google.com/?q=Jl.%20Pantai%20Balan...,NaN,NaN,https://www.airbnb.com/rooms/1229763427554175376,4.7,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,1,Villa
5,Surfer’s Villa 4 Mins to Bingin & Dreamland Be...,"Pecatu, Kuta Selatan, Badung, Bali, Nusa Tengg...",Beds: 2,Baths: 2.5,"Pool view, Villa",Starting from USD 118 per night,https://www.bukitvista.com/property/surfers-vi...,Bukit Vista,"Jl. Bangbang Metuug, Pecatu, Kec. Kuta Sel., K...",Bali,Bali,Pecatu,Indonesia,80361.0,4.0,"Long Stay Trend Property, New Listing, Private...","http://maps.google.com/?q=Pecatu,%20Kuta%20Sel...",-8.826667,115.114961,https://www.airbnb.com/rooms/12330187636979381...,4.57,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,Villa
6,Private Pool Villa Minutes from Bingin Surf Ha...,"Pecatu, Kuta Selatan, Badung, Bali, Nusa Tengg...",Beds: 2,Baths: 2,"Pool view, Villa",Starting from USD 165 per night,https://www.bukitvista.com/property/private-po...,Bukit Vista,"Jl. Bangbang Metuug, Pecatu, Kec. Kuta Sel., K...",Bali,Bali,Pecatu,Indonesia,80361.0,4.0,"Long Stay Trend Property, New Listing, Private...","http://maps.google.com/?q=Pecatu,%20Kuta%20Sel...",-8.826667,115.114961,https://www.airbnb.com/rooms/12330187636979381...,4.57,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,Villa
7,Ungasan Exquisite Villa w/ Rooftop & Private P...,"Ungasan, 

In [58]:
# Mengisisi nilai yang hiang dengan cara manual

df.at[33, 'rating'] = df['rating'].mode()[0]
df['rating'] = df['rating'].astype(float)

# manual imputation
df.at[45, 'price_range'] = 'Rp1,558,440 per night'
df.at[47, 'property_type'] = 'Villa'
df['country'] = df['country'].fillna('Indonesia')
df.at[31, 'area'] = 'Yogyakarta'
df.loc[0, ['city', 'state', 'area']] = ['Bali', 'Bali', 'Uluwatu']

df.loc[47, ['city', 'state', 'area']] = df[['city', 'state', 'area']].mode().iloc[0]

In [72]:
# Fungsi untuk mengekstrak informasi harga
def extract_price_range(price_string):
    # Pola regex untuk mata uang, harga, dan periode
    currency_pattern = r'(USD|Rp|[$])'
    price_pattern = r'(\d{1,3}(?:[.,]\d{3})*(?:\.\d+)?)'
    period_pattern = r'(per night|per month|per \d+ nights|per \d+ months|/Month|night|per Month|per Malam| / Night)'

    # Mencocokkan mata uang
    currency_match = re.search(currency_pattern, price_string)
    currency = currency_match.group(0) if currency_match else None

    # Mencocokkan harga
    price_match = re.search(price_pattern, price_string)
    price = price_match.group(0) if price_match else None

    # Mencocokkan periode
    period_match = re.search(period_pattern, price_string)
    period = period_match.group(0) if period_match else None

    return currency, price, period

In [74]:
# split from currency & period
df[['currency', 'price_value', 'period']] = df['price_range'].apply(lambda x: pd.Series(extract_price_range(x)))

KeyError: 'price_range'

In [70]:
df

,price_string,currency,price,period
0,Rp 1.200.000 per malam,Rp,1.200.000,None
1,$99.99 per night,$,99.99,per night
2,"USD 1,000 /Month",USD,"1,000",/Month


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               51 non-null     object 
 1   location           39 non-null     object 
 2   bedrooms           51 non-null     object 
 3   bathrooms          51 non-null     object 
 4   property_type      51 non-null     object 
 5   price_range        51 non-null     object 
 6   url                51 non-null     object 
 7   agency             43 non-null     object 
 8   detailed_address   49 non-null     object 
 9   city               51 non-null     object 
 10  state              51 non-null     object 
 11  area               51 non-null     object 
 12  country            51 non-null     object 
 13  zip_code           40 non-null     float64
 14  number_of_guests   43 non-null     float64
 15  property_status    46 non-null     object 
 16  maps_url           38 non-nu

In [60]:
df.isnull().sum()

name                  0
location             12
bedrooms              0
bathrooms             0
property_type         0
price_range           0
url                   0
agency                8
detailed_address      2
city                  0
state                 0
area                  0
country               0
zip_code             11
number_of_guests      8
property_status       5
maps_url             13
latitude             39
longitude            39
airbnb_url            1
rating                2
Amazing View          0
Amazing pool          0
Beachfront            0
Golfing               0
Guest House           0
Island life           0
Jungle View           0
Ocean view            0
Pool view             0
Residential           0
Rice paddy view       0
Style                 0
Surfing               0
Tropical              0
View                  0
Villa                 0
property_category     0
dtype: int64